## SHAP EXPLAINABILITY SUMMARY

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import os
import time

# --- Model loading ---
import joblib

# --- Explainability ---
import shap                           # SHAP values — the gold standard for tree explainability
import xgboost as xgb

# --- Pipeline extraction ---
from imblearn.pipeline import Pipeline as ImbPipeline   # needed to detect pipeline wrapping

# --- Drift detection ---
from river import drift as river_drift   # ADWIN and other streaming detectors

# --- Evaluation helpers ---
from sklearn.metrics import confusion_matrix, f1_score

warnings.filterwarnings("ignore")

# ── Reproducibility ──────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Plot style ───────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
FRAUD_PALETTE = {0: "#2196F3", 1: "#F44336"}   # Blue = legit, Red = fraud

# ── File paths ───────────────────────────────────────────────────────────────
PARQUET_TEST  = os.path.join("data", "test.parquet")
MODEL_LR      = os.path.join("models", "lr_baseline.joblib")
MODEL_XGB_SPW = os.path.join("models", "xgb_spw.joblib")
MODEL_XGB_SMO = os.path.join("models", "xgb_smote.joblib")
MODEL_META    = os.path.join("models", "model_meta.joblib")
FIGURES_DIR   = os.path.join("reports", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

# ── Stage 11 constants ────────────────────────────────────────────────────────
# TreeSHAP is fast (polynomial in tree depth, not exponential in features),
# but running it on 500K test rows is unnecessary and slow. 5,000 rows with
# stratified fraud oversampling captures the global distribution well while
# keeping SHAP runtime under ~30 seconds on a laptop.
N_SHAP_SAMPLE    = 5_000
N_FRAUD_IN_SHAP  = 2_000  # at most this many fraud rows in the SHAP sample

# ── Stage 12 constants ────────────────────────────────────────────────────────
# Inject drift 70% through the stream. This gives ADWIN a long "stable" window
# to calibrate on before the drift appears, which makes detection more reliable
# and produces a cleaner plot — you can clearly see the error-rate rise.
DRIFT_INJECT_FRAC = 0.70

# ADWIN sensitivity parameter (delta).
# Smaller delta → more sensitive, fires earlier, more false alarms.
# Larger delta → less sensitive, fires later, fewer false alarms.
# delta = 0.002 is a standard starting point for fraud error streams.
ADWIN_DELTA = 0.002

# Rolling window for the error-rate visualization (in number of transactions).
# Larger windows smooth the plot; smaller ones show finer-grained changes.
ROLLING_WINDOW = 1_000

print("    Imports and configuration complete.")
print(f"   SHAP sample size : {N_SHAP_SAMPLE:,} rows (up to {N_FRAUD_IN_SHAP:,} fraud)")
print(f"   Drift injection  : {DRIFT_INJECT_FRAC*100:.0f}% through stream  "
      f"·  ADWIN delta={ADWIN_DELTA}")

In [ ]:
for path in [PARQUET_TEST, MODEL_LR, MODEL_XGB_SPW, MODEL_XGB_SMO, MODEL_META]:
    assert os.path.exists(path), (
        f"\n  Missing: {path}\n"
        f"    → Run all cells in fraud_detection_stage9_10.ipynb first."
    )

# ── Load metadata ────────────────────────────────────────────────────────────
meta              = joblib.load(MODEL_META)
FEATURE_COLS      = meta['feature_cols']
DECISION_THRESHOLD = meta["DECISION_THRESHOLD"]
BEST_MODEL_NAME   = meta["BEST_MODEL_NAME"]

# ── Load test set ─────────────────────────────────────────────────────────────
# test.parquet contains step + FEATURE_COLS + isFraud (Stage 5/6 convention).
# Keeping 'test' as a full DataFrame so Stage 12 can sort by 'step'.
test   = pd.read_parquet(PARQUET_TEST)
X_test = test[FEATURE_COLS]
y_test = test["isFraud"].astype(int)

# ── Load best model ───────────────────────────────────────────────────────────
_model_path_map = {
    "LR Baseline" : MODEL_LR,
    "XGB (SPW)"   : MODEL_XGB_SPW,
    "XGB (SMOTE)" : MODEL_XGB_SMO,
}
assert BEST_MODEL_NAME in _model_path_map, (
    f"Unexpected BEST_MODEL_NAME='{BEST_MODEL_NAME}' in model_meta. "
    f"Expected one of {list(_model_path_map.keys())}."
)
best_model_raw = joblib.load(_model_path_map[BEST_MODEL_NAME])

# ── Extract native XGBoost model for TreeExplainer ────────────────────────────
# TreeExplainer maps directly to the tree structure of the booster.
# If the model is wrapped in an ImbPipeline (e.g., the SMOTE pipeline),
# we pull the XGBClassifier out of the named 'model' step.
if isinstance(best_model_raw, xgb.XGBClassifier):
    xgb_for_shap = best_model_raw
elif isinstance(best_model_raw, ImbPipeline):
    xgb_for_shap = best_model_raw.named_steps["model"]
    assert isinstance(xgb_for_shap, xgb.XGBClassifier), (
        "Expected the ImbPipeline 'model' step to be an XGBClassifier."
    )
else:
    raise TypeError(
        f"SHAP TreeExplainer requires an XGBClassifier. "
        f"Got {type(best_model_raw).__name__}. "
        f"If LR Baseline won (unexpected), use shap.LinearExplainer instead."
    )

print(f"Best model : {BEST_MODEL_NAME}")
print(f"Threshold  : {DECISION_THRESHOLD:.4f}")
print(f"Features   : {FEATURE_COLS}")
print()
print(f"Test set   : {X_test.shape[0]:,} rows × {X_test.shape[1]} features")
print(f"Fraud      : {y_test.sum():,} ({y_test.mean()*100:.4f}%)")
print()
print(f"xgb_for_shap type : {type(xgb_for_shap).__name__}  ✓")

In [ ]:
builtin_importance = pd.Series(
    xgb_for_shap.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=False)

print("Built-in XGBoost feature importance (F-score / gain):")
print(builtin_importance.round(4).to_string())

fig, ax = plt.subplots(figsize=(9, 5))
builtin_importance.plot(kind="barh", color="#FF5722", ax=ax)
ax.invert_yaxis()
ax.set_xlabel("Importance Score", fontsize=12)
ax.set_title(
    "Built-in XGBoost Feature Importance\n(treat as a rough first look only — SHAP is more reliable)",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "shap_builtin_importance.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print(f"\n Figure saved → {FIGURES_DIR}/shap_builtin_importance.png")
print("\nNote: Stage 6 explains why SHAP will give a more honest ranking.")

In [ ]:
print("Why SHAP?")
print()
print("  Key points:")
print("  • Shapley values are the unique fair credit-allocation that satisfies")
print("    four game-theory axioms: efficiency, symmetry, dummy, and linearity")
print("  • TreeSHAP gives EXACT values (not approximations) in polynomial time")
print("  • Local SHAP → justify one flagged transaction")
print("  • Global SHAP → rank features honestly, without training-data bias")

In [ ]:
import importlib
import shap.explainers._tree as _shap_tree

_fpath = inspect.getfile(_shap_tree)
if _fpath.endswith(".pyc"):
    _fpath = _fpath.replace("/__pycache__", "").rsplit(".", 1)[0] + ".py"

print(f"SHAP source: {_fpath}")

with open(_fpath, "r", encoding="utf-8") as _f:
    _src = _f.read()

# ── Target: the shared float() expression, independent of the assignment LHS ──
_OLD = 'float(learner_model_param["base_score"])'
_NEW = 'float(str(learner_model_param["base_score"]).strip("[]"))'

_n_remaining = _src.count(_OLD)

if _n_remaining == 0:
    # Also check if an earlier partial patch is still lurking
    if "_bs_raw" in _src:
        print("   Earlier _bs_raw patch detected but new target not found.")
        print("   The file may be in a mixed state. Checking for old patch...")
        # Check if the old partial patch is causing issues
        _OLD_BS_RAW = '_bs_raw = learner_model_param["base_score"]'
        if _OLD_BS_RAW in _src:
            print("   Found _bs_raw pattern — cleaning up old patch and applying fresh fix.")
            # Remove old _bs_raw lines and restore original for a clean replacement
            _src = _src.replace(
                '_bs_raw = learner_model_param["base_score"]\n',
                ''
            ).replace(
                'float(str(_bs_raw).strip("[]"))',
                'float(learner_model_param["base_score"])'
            )
            _n_remaining = _src.count(_OLD)
            print(f"   Restored {_n_remaining} original line(s) for clean replacement.")

if _n_remaining > 0:
    _new_src = _src.replace(_OLD, _NEW)    # replaces ALL occurrences
    _n_fixed = _src.count(_OLD)
    with open(_fpath, "w", encoding="utf-8") as _f:
        _f.write(_new_src)
    print(f"  Patched {_n_fixed} occurrence(s) of the broken float() call.")
elif _NEW in _src:
    print("   All occurrences already patched — nothing to change.")
else:
    raise RuntimeError(
        f"Could not find the target expression in {_fpath}.\n"
        "Open the file and search for every line containing:\n"
        '    float(learner_model_param["base_score"])\n'
        "Replace each with:\n"
        '    float(str(learner_model_param["base_score"]).strip("[]"))'
    )

# ── Reload so the fix is live without a kernel restart ───────────────────────
importlib.reload(_shap_tree)
import shap
shap.TreeExplainer = _shap_tree.TreeExplainer
print("  shap.TreeExplainer reloaded — all occurrences patched.")
print()

# ── Stratified sample ─────────────────────────────────────────────────────────
import numpy as np

rng         = np.random.default_rng(RANDOM_STATE)
idx_fraud   = y_test[y_test == 1].index.tolist()
idx_legit   = y_test[y_test == 0].index.tolist()

n_fraud_in_sample = min(len(idx_fraud), N_FRAUD_IN_SHAP)
n_legit_in_sample = N_SHAP_SAMPLE - n_fraud_in_sample

sample_fraud = rng.choice(idx_fraud, size=n_fraud_in_sample, replace=False)
sample_legit = rng.choice(idx_legit, size=n_legit_in_sample, replace=False)
sample_idx   = np.concatenate([sample_fraud, sample_legit])
rng.shuffle(sample_idx)

X_shap = X_test.loc[sample_idx].reset_index(drop=True)
y_shap = y_test.loc[sample_idx].reset_index(drop=True)

print(f"SHAP sample: {len(X_shap):,} rows  "
      f"({n_fraud_in_sample:,} fraud  |  {n_legit_in_sample:,} legit)")
print()

# ── TreeExplainer ─────────────────────────────────────────────────────────────
print("⏳ Building TreeExplainer and computing SHAP values...")
t0          = time.time()
explainer   = shap.TreeExplainer(xgb_for_shap)
shap_values = explainer.shap_values(X_shap)
elapsed     = time.time() - t0

if isinstance(shap_values, list):
    shap_values = shap_values[1]   # positive class (fraud)

base_value = explainer.expected_value
if hasattr(base_value, "__len__"):
    base_value = float(base_value[1])
else:
    base_value = float(base_value)

print(f"✅ SHAP values computed in {elapsed:.1f}s")
print(f"   shap_values shape : {shap_values.shape}  (rows × features)")
print(f"   base_value        : {base_value:.4f}  (model average in log-odds space)")
print()
print("   Sanity check: shap_values[i].sum() + base_value ≈ model log-odds for row i")

In [ ]:
mean_abs_shap = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=FEATURE_COLS
).sort_values(ascending=False)

print("Global SHAP importance (mean |SHAP value|):")
print(mean_abs_shap.round(5).to_string())

# ── Plot ──────────────────────────────────────────────────────────────────────
shap.summary_plot(shap_values, X_shap, plot_type="bar", show=False)
plt.title("Global Feature Importance — Mean |SHAP Value|", fontsize=13,
          fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "shap_bar_chart.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print(f"\n✅ Figure saved → {FIGURES_DIR}/shap_bar_chart.png")

# Store the top feature for use in the dependence plot (Cell 10) and drift
# injection (Cell 15) — will be the feature with the highest mean |SHAP|.
TOP_SHAP_FEATURE = mean_abs_shap.index[0]
print(f"\n   Top SHAP feature: '{TOP_SHAP_FEATURE}'  →  used in Cells 10 and 15")

In [ ]:
shap.summary_plot(shap_values, X_shap, show=False)
plt.title("SHAP Beeswarm — Feature Impact Direction and Magnitude", fontsize=13,
          fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "shap_beeswarm.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Figure saved → {FIGURES_DIR}/shap_beeswarm.png")
print()
print("Reading the Plot :")
print("  'errorBalanceOrig shows the clearest pattern: large values (red)")
print("   consistently push the fraud score right, confirming that the")
print("   accounting discrepancy is the model's primary fraud signal.'")

In [ ]:
# ── Fraud probability scores for the SHAP sample ─────────────────────────────
probs_shap = pd.Series(
    xgb_for_shap.predict_proba(X_shap)[:, 1],
    index=X_shap.index,
    name="fraud_prob",
)

# ── Find the highest-probability true positive ─────────────────────────────────
is_tp = (probs_shap >= DECISION_THRESHOLD) & (y_shap == 1)

assert is_tp.sum() > 0, (
    "No true positives found in the SHAP sample at the current threshold. "
    "Try lowering DECISION_THRESHOLD or increasing N_SHAP_SAMPLE."
)

tp_idx     = probs_shap[is_tp].idxmax()          # pandas label of highest-score TP
tp_row_pos = X_shap.index.get_loc(tp_idx)        # positional row in shap_values array
tp_prob    = probs_shap[tp_idx]

print(f"Explaining the model's most-confident caught fraud:")
print(f"  Sample index      : {tp_idx}")
print(f"  Fraud probability : {tp_prob:.4f}  (threshold = {DECISION_THRESHOLD:.4f})")
print(f"  True label        : FRAUD (1)")
print()
print("  Feature values for this transaction:")
print(X_shap.loc[tp_idx].to_string())

# ── Waterfall plot ────────────────────────────────────────────────────────────
shap_exp_tp = shap.Explanation(
    values        = shap_values[tp_row_pos],
    base_values   = base_value,
    data          = X_shap.iloc[tp_row_pos].values,
    feature_names = list(X_shap.columns),
)
shap.plots.waterfall(shap_exp_tp, show=False)
plt.title(f"SHAP Waterfall — Caught Fraud  (P(fraud)={tp_prob:.3f})",
          fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "shap_waterfall_caught_fraud.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print(f"\n Figure saved → {FIGURES_DIR}/shap_waterfall_caught_fraud.png")
print()
print("Reading this plot: bars pointing right (red) pushed the score toward fraud.")
print("The sum of all bars + base_value ≈ the model's log-odds output.")

In [ ]:
is_fn = (probs_shap < DECISION_THRESHOLD) & (y_shap == 1)

if is_fn.sum() == 0:
    print("  No false negatives in the current SHAP sample.")
    print("   This means the model catches ALL fraud in this sample at the chosen threshold.")
    print("   Options:")
    print("     1. Raise DECISION_THRESHOLD temporarily to create some FNs for illustration")
    print("     2. Increase N_SHAP_SAMPLE so more edge-case fraud rows are included")
    print("     3. Interpret this as the model performing very well (PaySim is synthetic)")
else:
    # Pick the FN closest to the threshold — the "most interesting" case
    fn_scores      = probs_shap[is_fn]
    fn_idx         = (fn_scores - DECISION_THRESHOLD).abs().idxmin()
    fn_row_pos     = X_shap.index.get_loc(fn_idx)
    fn_prob        = probs_shap[fn_idx]

    print(f"Explaining the model's closest-to-threshold missed fraud:")
    print(f"  Sample index      : {fn_idx}")
    print(f"  Fraud probability : {fn_prob:.4f}  (threshold = {DECISION_THRESHOLD:.4f})")
    print(f"  True label        : FRAUD (1)  ← but the model said 'clear'")
    print()
    print("  Feature values for this transaction:")
    print(X_shap.loc[fn_idx].to_string())

    shap_exp_fn = shap.Explanation(
        values        = shap_values[fn_row_pos],
        base_values   = base_value,
        data          = X_shap.iloc[fn_row_pos].values,
        feature_names = list(X_shap.columns),
    )
    shap.plots.waterfall(shap_exp_fn, show=False)
    plt.title(
        f"SHAP Waterfall — Missed Fraud / False Negative  (P(fraud)={fn_prob:.3f})\n"
        f"Real fraud the model CLEARED — understanding this guides model improvement",
        fontsize=11, fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, "shap_waterfall_missed_fraud.png"),
                dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\n Figure saved → {FIGURES_DIR}/shap_waterfall_missed_fraud.png")
    print()
    print("Reading this plot: look for bars pointing LEFT (blue). These features")
    print("suppressed the fraud score below the threshold. Compare the feature")
    print("values here against the caught-fraud case (Cell 8) — the differences")
    print("show what made this fraud 'look' more legitimate to the model.")

In [ ]:
dep_feature = TOP_SHAP_FEATURE
if dep_feature not in X_shap.columns:
    dep_feature = X_shap.columns[np.abs(shap_values).mean(axis=0).argmax()]
    print(f"Note: using '{dep_feature}' (highest SHAP importance)")

print(f"Dependency plot for: '{dep_feature}'")
print(f"  (y-axis = SHAP value, x-axis = feature value, color = auto-interaction)")

shap.dependence_plot(dep_feature, shap_values, X_shap,
                     interaction_index="auto", show=False)
plt.title(
    f"SHAP Dependency Plot — '{dep_feature}'\n"
    f"How this feature's value drives the fraud prediction",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(
    os.path.join(FIGURES_DIR, f"shap_dependence_{dep_feature.replace('/', '_')}.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()
print(f"\n✅ Figure saved → {FIGURES_DIR}/shap_dependence_*.png")

In [ ]:
print("Why explainability matters in Fintech?")
print()
print("  Three compounding reasons:")
print("  1. Regulatory compliance — per-decision justification may be legally required")
print("  2. Fairness auditing    — inspect what the model is actually keying on")
print("  3. Analyst trust        — investigators act on reasons, not scores alone")
print()
print()
print("  'I included SHAP explainability not as an academic add-on but because")
print("   Fintech deployments increasingly require per-prediction justification")
print("   for compliance. The waterfall plots in this project are what a fraud")
print("   analyst would see when reviewing a flagged transaction.'")

In [ ]:
meta = joblib.load(MODEL_META)
meta["SHAP_FEATURE_RANKING"] = mean_abs_shap.index.tolist()
meta["SHAP_MEAN_ABS_VALUES"] = {k: round(float(v), 6)
                                 for k, v in mean_abs_shap.items()}
meta["TOP_SHAP_FEATURE"]     = TOP_SHAP_FEATURE
joblib.dump(meta, MODEL_META)

print("=" * 62)
print("  STAGE 11 COMPLETE — SHAP EXPLAINABILITY SUMMARY")
print("=" * 62)
print()
print("  Global SHAP feature ranking (most → least important):")
for i, (feat, val) in enumerate(mean_abs_shap.items(), start=1):
    print(f"    {i}. {feat:<30}  mean |SHAP| = {val:.5f}")
print()
print("  Local explanations produced:")
print("    • Caught fraud waterfall    (Cell 8)  — primary demo for portfolio")
print("    • Missed fraud waterfall    (Cell 9)  — shows model failure analysis")
print("    • Dependency plot           (Cell 10) — validates feature engineering")
print()
print("  model_meta.joblib updated with SHAP rankings.")
print()
print("  NEXT: Stage 12 — Concept Drift Monitoring with ADWIN (River)")
print("=" * 62)

## Concept Drift Monitoring with ADWIN

In [ ]:
print("Stage 12 — Concept Drift Monitoring")
print()
print("  Three drift types:")
print("  1. Covariate drift  — X distribution shifts (new patterns, inflation)")
print("  2. Prior drift      — fraud rate changes (bust/boom of fraud rings)")
print("  3. Concept drift    — X→y relationship changes (fraudsters adapt)  ← most dangerous")
print()
print("  Detector: ADWIN (River library)")
print("    · Maintains a variable-length window over the model's error stream")
print("    · Fires when old-window mean error ≠ new-window mean error (Hoeffding bound)")
print("    · Self-tuning window size — no manual window width to set")
print(f"    · delta={ADWIN_DELTA}  (sensitivity parameter; lower = more sensitive)")

In [ ]:
# Sort by step for temporal ordering
if "step" in test.columns:
    sort_order = test["step"].argsort().values
    print("Sorting test stream by 'step' (simulation time order)")
else:
    sort_order = np.arange(len(test))
    print("'step' not in test.parquet — using row order (temporal from Stage 6 split)")

X_stream = X_test.iloc[sort_order].reset_index(drop=True)
y_stream = y_test.iloc[sort_order].reset_index(drop=True)
steps    = (test["step"].iloc[sort_order].reset_index(drop=True)
            if "step" in test.columns else pd.Series(np.arange(len(test))))

DRIFT_INJECT_IDX = int(len(X_stream) * DRIFT_INJECT_FRAC)
n_before = DRIFT_INJECT_IDX
n_after  = len(X_stream) - DRIFT_INJECT_IDX
n_fraud_before = int(y_stream.iloc[:DRIFT_INJECT_IDX].sum())
n_fraud_after  = int(y_stream.iloc[DRIFT_INJECT_IDX:].sum())

print()
print(f"Stream summary:")
print(f"  Total transactions    : {len(X_stream):,}")
print(f"  Drift injection at    : index {DRIFT_INJECT_IDX:,}  "
      f"({DRIFT_INJECT_FRAC*100:.0f}% through stream)")
print(f"  Before injection      : {n_before:,} rows  ({n_fraud_before:,} fraud)")
print(f"  After injection       : {n_after:,} rows  ({n_fraud_after:,} fraud)")
print(f"  Rolling window        : {ROLLING_WINDOW:,} transactions")

In [ ]:
DRIFT_FEATURE = TOP_SHAP_FEATURE   # the feature identified in Cell 6 as most important
if DRIFT_FEATURE not in X_stream.columns:
    DRIFT_FEATURE = FEATURE_COLS[0]   # safe fallback
    print(f"  '{TOP_SHAP_FEATURE}' not found in stream; falling back to '{DRIFT_FEATURE}'")

# Apply the drift modification
X_drifted        = X_stream.copy()
after_inject_mask = X_stream.index >= DRIFT_INJECT_IDX
fraud_mask        = y_stream == 1
drift_target_mask = after_inject_mask & fraud_mask

X_drifted.loc[drift_target_mask, DRIFT_FEATURE] = (
    -X_drifted.loc[drift_target_mask, DRIFT_FEATURE]
)

n_modified = int(drift_target_mask.sum())
print(f"Drift feature      : '{DRIFT_FEATURE}'")
print(f"Modification       : negate '{DRIFT_FEATURE}' for fraud rows after index {DRIFT_INJECT_IDX}")
print(f"Rows modified      : {n_modified:,} fraud transactions")
print()
print(f"Effect: for these {n_modified:,} rows, the model's strongest fraud signal")
print(f"  now points in the WRONG direction → error rate rises → ADWIN fires.")
print()
print("What this simulates in business terms:")
print("  Fraudsters learned that large balance discrepancies trigger the system.")
print("  They now execute their moves in smaller steps that cancel out,")
print("  making the books appear balanced. The model hasn't seen this tactic.")

In [ ]:
print("⏳ Batch-predicting on original and drifted streams...")
t0 = time.time()

probs_original  = xgb_for_shap.predict_proba(X_stream)[: , 1]
probs_drifted   = xgb_for_shap.predict_proba(X_drifted)[:, 1]

preds_original  = (probs_original >= DECISION_THRESHOLD).astype(int)
preds_drifted   = (probs_drifted  >= DECISION_THRESHOLD).astype(int)

errors_original = (preds_original != y_stream.values).astype(int)
errors_drifted  = (preds_drifted  != y_stream.values).astype(int)

print(f" Predictions complete in {time.time()-t0:.1f}s")
print()
print(f"Baseline error rate (no drift)  : {errors_original.mean()*100:.4f}%")
print(f"Drifted  error rate (full stream): {errors_drifted.mean()*100:.4f}%")
print(f"Error rate AFTER injection only  : "
      f"{errors_drifted[DRIFT_INJECT_IDX:].mean()*100:.4f}%")
print()

# ── Stream through ADWIN ──────────────────────────────────────────────────────
print(f" Streaming {len(errors_drifted):,} errors through ADWIN (delta={ADWIN_DELTA})...")
t0 = time.time()

# Baseline — original (no drift)
adwin_baseline      = river_drift.ADWIN(delta=ADWIN_DELTA)
drift_events_baseline = []
for i, err in enumerate(errors_original):
    adwin_baseline.update(int(err))
    if adwin_baseline.drift_detected:
        drift_events_baseline.append(i)

# Drifted stream
adwin_drifted      = river_drift.ADWIN(delta=ADWIN_DELTA)
drift_events_drifted = []
for i, err in enumerate(errors_drifted):
    adwin_drifted.update(int(err))
    if adwin_drifted.drift_detected:
        drift_events_drifted.append(i)

print(f" ADWIN streaming complete in {time.time()-t0:.1f}s")
print()
print(f"Baseline  (no drift)  : {len(drift_events_baseline)} detection event(s)")
print(f"Drifted stream        : {len(drift_events_drifted)} detection event(s)")

if drift_events_drifted:
    first_detection = drift_events_drifted[0]
    detection_lag   = first_detection - DRIFT_INJECT_IDX
    print(f"  First detection at index  : {first_detection:,}")
    print(f"  Injection was at index    : {DRIFT_INJECT_IDX:,}")
    print(f"  Detection lag             : {detection_lag:,} transactions after drift onset")

In [ ]:
rolling_orig   = pd.Series(errors_original.astype(float)).rolling(ROLLING_WINDOW).mean() * 100
rolling_drift  = pd.Series(errors_drifted.astype(float)).rolling(ROLLING_WINDOW).mean()  * 100
x_axis         = np.arange(len(errors_drifted))

fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(x_axis, rolling_orig, color="#2196F3", linewidth=1.5, alpha=0.7,
        label=f"Original stream — no drift  (baseline)")
ax.plot(x_axis, rolling_drift, color="#FF5722", linewidth=2.0,
        label=f"Drifted stream — '{DRIFT_FEATURE}' negated for fraud after injection")

# Drift injection marker
ax.axvline(x=DRIFT_INJECT_IDX, color="#E91E63", linestyle="--", linewidth=2.5,
           label=f"Drift injected  (index {DRIFT_INJECT_IDX:,})")

# ADWIN alert markers on drifted stream
if drift_events_drifted:
    for j, ev in enumerate(drift_events_drifted):
        ax.axvline(x=ev, color="#F44336", linestyle=":", linewidth=1.2, alpha=0.6)
    # Label just the first alert in the legend
    ax.axvline(x=drift_events_drifted[0], color="#F44336", linestyle=":", linewidth=1.5,
               alpha=0.8,
               label=f"ADWIN alert  ({len(drift_events_drifted)} total, "
                     f"first at index {drift_events_drifted[0]:,})")

# ADWIN alerts on baseline (usually none)
if drift_events_baseline:
    for ev in drift_events_baseline:
        ax.axvline(x=ev, color="#9C27B0", linestyle=":", linewidth=1.0, alpha=0.5)
    ax.axvline(x=drift_events_baseline[0], color="#9C27B0", linestyle=":", linewidth=1.5,
               alpha=0.8,
               label=f"ADWIN baseline alert  ({len(drift_events_baseline)} total)")

ax.set_xlabel(f"Stream position (transactions)", fontsize=12)
ax.set_ylabel(f"Rolling error rate %  (window = {ROLLING_WINDOW:,})", fontsize=12)
ax.set_title(
    f"Concept Drift Monitoring with ADWIN  [delta={ADWIN_DELTA}]\n"
    f"'{DRIFT_FEATURE}' negated for fraud after index {DRIFT_INJECT_IDX:,}",
    fontsize=13, fontweight="bold"
)
ax.legend(fontsize=10, loc="upper left")
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "drift_error_stream.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print(f" Figure saved → {FIGURES_DIR}/drift_error_stream.png")

if drift_events_drifted:
    print()
    print(f"   ADWIN detected drift  {detection_lag:,} transactions after it was injected.")
    print(f"     In a 30-day PaySim simulation, this lag ≈ "
          f"{detection_lag / (len(X_stream) / 30):.1f} simulated days.")
else:
    print()
    print("     ADWIN did not fire on the drifted stream.")
    print("     Options to investigate:")
    print("     1. Verify the drift feature is actually in FEATURE_COLS")
    print("     2. Reduce ADWIN_DELTA to increase sensitivity")
    print("     3. Increase DRIFT_INJECT_FRAC to give ADWIN more post-drift data")


In [ ]:
print("Drift response framework.")
print()
print("  DETECT   → ADWIN fires, log the event")
print("  DIAGNOSE → Is it real drift, a pipeline bug, or a transient spike?")
print("  DECIDE   → If drift is real + fresh labels available → retrain")
print("             Otherwise → alert team, tighten threshold, await labels")
print()
print("  For this project:")
print("  → We log the detection events and surface them in the Streamlit")
print("    dashboard as red alert markers on the drift panel.")
print("  → Retrain logic is out of scope;")
print()
print("  One key production nuance is:")
print("  'In live fraud detection, labels arrive with a delay — typically")
print("   days after the transaction. ADWIN can detect that error rates")
print("   are rising even before labels arrive, because it can monitor")
print("   model confidence distributions or proxy signals. Full supervised")
print("   retraining requires waiting for confirmed labels.'")

if drift_events_drifted:
    print()
    print(f"  Drift events on the drifted stream (indices):")
    for ev in drift_events_drifted[:10]:   # show at most 10
        print(f"    index {ev:,}  "
              f"(rolling error at this point: {rolling_drift.iloc[ev]:.2f}%)")
    if len(drift_events_drifted) > 10:
        print(f"    ... and {len(drift_events_drifted) - 10} more")

In [ ]:
meta = joblib.load(MODEL_META)

meta["DRIFT_FEATURE"]           = DRIFT_FEATURE
meta["DRIFT_INJECT_IDX"]        = int(DRIFT_INJECT_IDX)
meta["DRIFT_INJECT_FRAC"]       = float(DRIFT_INJECT_FRAC)
meta["ADWIN_DELTA"]             = float(ADWIN_DELTA)
meta["DRIFT_EVENTS_DRIFTED"]    = [int(e) for e in drift_events_drifted]
meta["DRIFT_EVENTS_BASELINE"]   = [int(e) for e in drift_events_baseline]
meta["DRIFT_ROLLING_WINDOW"]    = int(ROLLING_WINDOW)

if drift_events_drifted:
    meta["DRIFT_FIRST_DETECTION_IDX"] = int(drift_events_drifted[0])
    meta["DRIFT_DETECTION_LAG"]       = int(drift_events_drifted[0] - DRIFT_INJECT_IDX)
else:
    meta["DRIFT_FIRST_DETECTION_IDX"] = None
    meta["DRIFT_DETECTION_LAG"]       = None

joblib.dump(meta, MODEL_META)

print(f" model_meta.joblib updated → {MODEL_META}")
print()
print("  Drift fields added:")
print(f"    DRIFT_FEATURE           : {DRIFT_FEATURE}")
print(f"    DRIFT_INJECT_IDX        : {DRIFT_INJECT_IDX:,}")
print(f"    ADWIN_DELTA             : {ADWIN_DELTA}")
print(f"    DRIFT_EVENTS_DRIFTED    : {len(drift_events_drifted)} event(s) "
      f"at indices {drift_events_drifted[:5]}{'...' if len(drift_events_drifted) > 5 else ''}")
print(f"    DRIFT_EVENTS_BASELINE   : {len(drift_events_baseline)} event(s)")
if drift_events_drifted:
    print(f"    DRIFT_DETECTION_LAG     : {drift_events_drifted[0] - DRIFT_INJECT_IDX:,} transactions")


In [ ]:
# =============================================================================
# %% CELL 20 — Stage 12 Checkpoint
# =============================================================================

print("=" * 62)
print("  STAGE 12 COMPLETE — CONCEPT DRIFT SUMMARY")
print("=" * 62)
print()
print(f"  Drift simulation:")
print(f"    Stream length          : {len(X_stream):,} transactions")
print(f"    Drift feature          : '{DRIFT_FEATURE}'")
print(f"    Injection point        : index {DRIFT_INJECT_IDX:,}  "
      f"({DRIFT_INJECT_FRAC*100:.0f}% through)")
print(f"    Rows modified          : {int(drift_target_mask.sum()):,} fraud transactions")
print()
print(f"  ADWIN results (delta={ADWIN_DELTA}):")
print(f"    Baseline alerts        : {len(drift_events_baseline)}  "
      f"(model stable without drift)")
print(f"    Drifted stream alerts  : {len(drift_events_drifted)}")
if drift_events_drifted:
    print(f"    First detection lag    : {drift_events_drifted[0] - DRIFT_INJECT_IDX:,} "
          f"transactions after injection")
print()
print(f"  Figures saved:")
print(f"    {FIGURES_DIR}/drift_error_stream.png")
print()
print(f"  model_meta.joblib updated with drift simulation results.")
print()
print("    'I demonstrated concept drift monitoring using ADWIN on a")
print("     PaySim test stream with artificially injected drift. ADWIN")
print(f"     detected the change within {meta.get('DRIFT_DETECTION_LAG', 'N/A')} "
         "transactions of injection.")
print("     In production, the Streamlit dashboard displays this plot live,")
print("     and the response protocol is: Detect → Diagnose → Decide,")
print("     not automatic retraining.'")
print()
print("  NEXT: Stage 13 — Model Artifact Packaging (for Streamlit)")
print("  (bundle model + metadata + feature order into one clean load)")
print("=" * 62)
